# 2026 FIFA World Cup Prediction
### Direct Translation Approach (Adapted from 2022 Kaggle Methodology)

This notebook demonstrates the end-to-end process of predicting the 2026 World Cup outcomes using the expanded 48-team format. We utilize a Gradient Boosting Classifier trained on historical match data and FIFA rankings from the 2026 cycle (2022-2026).

## 1. Data Ingestion & Preprocessing
We load the international match results and historical FIFA rankings, filtering for the 2026 cycle starting after the 2022 World Cup final.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os

results = pd.read_csv('../data/results.csv')
rankings = pd.read_csv('../data/fifa_ranking.csv')

results['date'] = pd.to_datetime(results['date'])
rankings['date'] = pd.to_datetime(rankings['date'])

# Calculate Rank if missing
if 'rank' not in rankings.columns:
    rankings = rankings.sort_values(['date', 'total_points'], ascending=[True, False])
    rankings['rank'] = rankings.groupby('date')['total_points'].rank(ascending=False, method='min')

cycle_start = pd.to_datetime('2022-12-19')
results_cycle = results[results['date'] >= cycle_start].copy()
rankings_cycle = rankings[rankings['date'] >= cycle_start].copy()

print(f"Matches in 2026 Cycle: {len(results_cycle)}")

## 2. Feature Engineering
We compute rolling goal averages (last 5 games) and extract FIFA ranks for each matchup to create the training features: `rank_diff`, `average_rank`, `point_diff`, and `goals_rolling_diff`.

In [ ]:
# Load final features generated in the implementation phase
final_data = pd.read_csv('../data/features/final_features.csv')
features = ['rank_diff', 'average_rank', 'point_diff', 'is_friendly', 'goals_rolling_diff']
final_data = final_data.dropna(subset=features + ['target'])

print("Feature Preview:")
display(final_data[features + ['target']].head())

## 3. Model Training & Validation
We use an optimized Gradient Boosting Classifier. The model predicts the probability of a Home team victory.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

with open('../models/optimized_gb.pkl', 'rb') as f:
    model = pickle.load(f)

print(f"Model Type: {type(model).__name__}")

## 4. 2026 World Cup Simulation
We simulate the entire tournament, from the 12 groups of 4 to the 32-team knockout bracket.

In [ ]:
from simulation_engine import run_simulation

# Setting working directory context to find data files
os.chdir('..')
group_results, best_thirds = run_simulation()
os.chdir('notebooks')

## Conclusion
The 2026 simulation confirms that the expanded 48-team format maintains high competitive integrity, with traditional powerhouses emerging in the later stages. In this specific run, **Argentina** is predicted to defend their title.